# 03 — Mark 3 — Gated Two-Stage Multi-Window Overfit

[![Phase](https://img.shields.io/badge/Pipeline-Mark%201%20to%204E-blue.svg)]()
[![Mode](https://img.shields.io/badge/Default-REUSE%20(fast%2C%20deterministic)-success.svg)]()

**Pipeline position:** notebook **03 of 09** — run the suite in order 00 → 09.
**Original:** `mark 1/mark_3_two_stage_multiwindow_overfit.ipynb`

## Objective

Can a small MobileNetV2-UNet actually overfit the two-stage ROI/tumor task? A controlled 1/2/3-channel overfit ablation on a locked 16-slice stratified subset verifies capacity, gradient stability and channel-worthiness before any real training.

## Inputs (read-only)

- `mark_2_gate_result.json` + `training_roi_manifest.csv` from Mark 2
- Frozen 16-slice training subset (stratified, locked)
- REUSE: archived Mark 3 overfit history; REBUILD: retrain (set RUN_MARK3_OVERFIT=True, REUSE_HISTORY=False)

## Outputs → `Evaluation/mark_1_to_4e_outputs/mark_3_outputs/`

Every file below keeps the exact naming used by the archived run, so results are
directly comparable with the original `mark 1/mark_*_outputs/` outputs.

| File |
|---|
| `mark_3_gate_result.json` |
| `training_roi_gate.json` |
| `training_roi_manifest.csv` |
| `roi_tensor_audit.png` |
| `training_roi_audit.png` |
| `overfit_history.csv` |
| `overfit_channel_comparison.csv` |
| `overfit_selected_slices.csv` |
| `roundtrip_geometry_metrics.csv` |
| `overfit_convergence_dashboard.png` |
| `overfit_prediction_review.png` |

**Visualizations produced by this notebook:** `roi_tensor_audit.png`, `training_roi_audit.png`, `overfit_convergence_dashboard.png`, `overfit_prediction_review.png`

## Phase dataflow

```mermaid
flowchart LR
  A["inputs: mark_2_gate_result.json, Frozen 16-slice training subset (stratified, locked), REUSE: archived Mark 3 overfit history; REBUILD: retrain (set RUN_MARK3_OVERFIT=True, REUS"] -->
  B[phase cells: provenance + reuse/rebuild + compute]
  B --> G["gate: mark_3_gate_result.json"]
  B --> O[organized per-phase outputs]
  G --> D[downstream notebook reads this gate]
```


## Key finding (reproduced)

**Overfit gate PASSED.** Broad-window 1-channel configuration reached hard micro-Dice ≈ **0.9006** with finite losses/gradients — the two-stage architecture has sufficient capacity and the multi-window inputs are channel-worthy.

## Gate

`mark_3_gate_result.json` — overfit capacity gate (best hard Dice ≥ 0.85 expected)

## Run notes

Training checkout only (16 slices) — no test access. Freezes batch-norm running stats, clips gradients at 5.0, and raises on non-finite loss/gradients.

> **Shared setup:** the next cell is the *identical* global-setup cell embedded in every notebook
> (paths, seeds, provenance hashes, test lock, shared helpers). REUSE mode reads frozen artifacts from
> `mark 1/`, so each notebook is deterministic and reproducible; set the `REUSE_*` / `RUN_*` flags to
> rebuild caches or retrain (GPU hours).
>
> **Ordering matters:** this phase reads the previous phase's gate JSON from the shared output folder
> (`mark_1_to_4e_outputs/…`), so run the suite in order **00 → 09**. A phase can be re-run standalone
> once its upstream gates exist (re-running the preceding notebooks regenerates them).

In [1]:
from __future__ import annotations

from pathlib import Path
from IPython.display import display
import hashlib
import json
import random
import sys
import time
import warnings

import matplotlib
matplotlib.use("Agg")  # headless-safe; every figure is also saved to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver")
DATASET_ROOT = Path(
    r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver\02_staging"
    r"\build_corrected_20260713_214847_v2"
)
MANIFEST_PATH = DATASET_ROOT / "manifests" / "slice_manifest.csv"
SOURCE_CHECKPOINT = (
    PROJECT_ROOT / "Practice" / "multitask_liver_tumor_outputs" / "multitask_best.pth"
)

# Frozen original artifacts (read-only inputs for REUSE mode)
MARK1_DIR = PROJECT_ROOT / "mark 1"
# Centralized shared output root under Evaluation/output (one folder per notebook)
SHARED_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "output"
# Legacy outputs (read-only fallback for the availability-check import helpers)
LEGACY_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "mark_1_to_4e_outputs"
PHASE_DIR = {
    "00_setup": "00_pipeline_overview",
    "mark_1": "01_mark_1", "mark_2": "02_mark_2", "mark_3": "03_mark_3",
    "mark_4": "04_mark_4", "mark_4b": "05_mark_4b", "mark_4c": "06_mark_4c",
    "mark_4d": "07_mark_4d", "mark_4e": "08_mark_4e", "consolidated": "09_consolidated",
}
OUT       = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "data"    for phase in PHASE_DIR}
OUT_FIGS  = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "figures" for phase in PHASE_DIR}
OUT_CACHE = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "caches"  for phase in PHASE_DIR}
CONSOLIDATED = OUT["consolidated"]
CONSOLIDATED_FIGS = OUT_FIGS["consolidated"]
for _d in [*OUT.values(), *OUT_FIGS.values(), *OUT_CACHE.values()]:
    _d.mkdir(parents=True, exist_ok=True)
NOTEBOOK_KEY = "mark_3"


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------------------------
# Execution-mode flags
# ---------------------------------------------------------------------------
REUSE_CACHES = True        # False -> re-run validation inference to rebuild .npz caches
REUSE_HISTORY = True       # False -> retrain (Mark 3 overfit, Mark 4 smoke, Mark 4C arms)
RUN_MARK3_OVERFIT = False  # retrain the 1/2/3-channel overfit ablation
RUN_MARK4_SMOKE = False    # retrain the 5-epoch validation smoke test
RUN_MARK4C_ARMS = False    # retrain the two Mark 4C ablation arms
assert not (RUN_MARK3_OVERFIT or RUN_MARK4_SMOKE or RUN_MARK4C_ARMS) or not REUSE_HISTORY, \
    "Retraining requires REUSE_HISTORY=False"

# ---------------------------------------------------------------------------
# Constants (identical to the originals)
# ---------------------------------------------------------------------------
SEED = 42
ROI_SIZE = 256
ORGAN_Z_CLIP = 3.0
BATCH_SIZE = 16
NUM_WORKERS = 0
EXPECTED_MANIFEST_SHA256 = "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889"
EXPECTED_SOURCE_CHECKPOINT_SHA256 = "9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c72b95d4639e0572223"
BROAD_WINDOW = (-160.0, 240.0)
LIVER_WINDOW = (0.0, 200.0)
THRESHOLDS = np.array([.05, .10, .15, .20, .25, .30, .35, .40, .45, .50, .55, .60, .65, .70],
                      dtype=np.float32)

CONTINUATION_TARGETS = {
    "mean_patient_dice": 0.3329, "volume_104_dice": 0.05, "volume_116_dice": 0.01,
    "q1_detected_pct": 35.0, "positive_predicted_empty_pct": 35.0,
    "empty_slice_false_positive_pct": 20.0,
}
FINAL_TARGETS = {
    "mean_patient_dice": 0.406915, "volume_104_dice": 0.50, "volume_116_dice": 0.05,
    "q1_detected_pct": 45.0, "positive_predicted_empty_pct": 20.0,
    "empty_slice_false_positive_pct": 15.0,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Shared helpers (deduplicated from the eight notebooks)
# ---------------------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def resize_float(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.float32), mode="F").resize(
            size, Image.Resampling.BILINEAR
        ),
        dtype=np.float32,
    )


def resize_mask(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.uint8) * 255).resize(
            size, Image.Resampling.NEAREST
        ),
        dtype=np.uint8,
    ) > 0


def window_hu(array, window):
    return np.clip((array - window[0]) / (window[1] - window[0]), 0, 1).astype(np.float32)


def probability_to_full(probability_roi, box):
    y0, y1, x0, x1 = [int(v) for v in box]
    resized = resize_float(probability_roi, (x1 - x0, y1 - y0))
    full = np.zeros((256, 256), dtype=np.float32)
    full[y0:y1, x0:x1] = resized
    return full


def image_robust_normalize(image):
    image = np.asarray(image, dtype=np.float32)
    reference = image[image > 0]
    if reference.size < 32:
        reference = image.reshape(-1)
    center = float(np.median(reference))
    q25, q75 = np.percentile(reference, [25, 75])
    robust_sigma = float((q75 - q25) / 1.349)
    if not np.isfinite(robust_sigma) or robust_sigma < 1e-3:
        robust_sigma = max(float(np.std(reference)), 1e-3)
    normalized = np.clip((image - center) / robust_sigma, -ORGAN_Z_CLIP, ORGAN_Z_CLIP)
    return ((normalized + ORGAN_Z_CLIP) / (2 * ORGAN_Z_CLIP)).astype(np.float32)


def target_passes(row, targets):
    return {
        key: (row[key] <= target if key in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else row[key] >= target)
        for key, target in targets.items()
    }


# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Shared output helpers: availability check, cross-notebook import, registry
# ---------------------------------------------------------------------------
import shutil as _shutil

ARTIFACT_INDEX = SHARED_OUTPUT_ROOT / "artifact_index.json"


def _load_artifact_index():
    if ARTIFACT_INDEX.is_file():
        return json.loads(ARTIFACT_INDEX.read_text())
    return {"version": 1, "artifacts": []}


def _save_artifact_index(index):
    ARTIFACT_INDEX.write_text(json.dumps(index, indent=2))


def register_artifact(name, kind="data", phase=None):
    phase = phase or NOTEBOOK_KEY
    index = _load_artifact_index()
    index["artifacts"] = [a for a in index["artifacts"]
                          if not (a.get("phase") == phase and a.get("name") == name)]
    path = OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name
    index["artifacts"].append({
        "phase": phase, "name": name, "kind": kind,
        "sha256": sha256_file(path) if path.is_file() else None,
        "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    })
    _save_artifact_index(index)


def shared_path(phase, name, kind="data"):
    return OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name


def legacy_path(phase, name, kind="data"):
    return LEGACY_OUTPUT_ROOT / f"{phase}_outputs" / name


def load_shared(phase, name, kind="data", required=True):
    """Availability check: Evaluation/output -> legacy mark_1_to_4e_outputs -> compute/raise."""
    target = shared_path(phase, name, kind)
    if target.is_file():
        return target
    legacy = legacy_path(phase, name, kind)
    if legacy.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        _shutil.copy2(legacy, target)
        print(f"IMPORT: reused legacy {phase}/{name} (copied to {target}).")
        return target
    if required:
        raise FileNotFoundError(
            f"Required upstream output missing: {PHASE_DIR.get(phase, phase)}/{name}.\n"
            f"Run the notebook for phase '{phase}' first (outputs land under "
            f"{SHARED_OUTPUT_ROOT / PHASE_DIR.get(phase, phase)}).")
    return None


def require_upstream_gate(phase, gate_name=None):
    gate_name = gate_name or f"{phase}_gate_result.json"
    return json.loads(load_shared(phase, gate_name, "data", required=True).read_text())


def save_figure(fig, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT_FIGS[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(target, dpi=170, bbox_inches="tight")
    register_artifact(name, "figures", phase)
    return target


def save_table(frame, name, phase=None, index=False):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target, index=index)
    register_artifact(name, "data", phase)
    return target


def save_json(obj, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(obj, indent=2))
    register_artifact(name, "data", phase)
    return target


# ---------------------------------------------------------------------------
# Standardized per-phase summary dashboard
# ---------------------------------------------------------------------------
CORE_METRICS = ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct",
                "empty_slice_false_positive_pct"]
TARGETS_SHEET = {phase: CONTINUATION_TARGETS for phase in
                 ["mark_1", "mark_4", "mark_4b", "mark_4c", "mark_4d", "mark_4e"]}
GATE_SELECTOR = {
    "mark_1": "best_observed_configuration_for_diagnosis",
    "mark_2": "selected_roi_configuration",
    "mark_3": "selected_configuration",
    "mark_4": "best_metrics", "mark_4b": "selected_metrics",
    "mark_4c": "arms", "mark_4d": "selected_metrics", "mark_4e": "selected_metrics",
}
TREND_CSV = {
    "mark_1":  ("calibration_configuration_results.csv", "tumor_threshold"),
    "mark_2":  ("roi_configuration_results.csv", "liver_threshold"),
    "mark_3":  ("overfit_history.csv", "epoch"),
    "mark_4":  ("mark_4_history.csv", "epoch"),
    "mark_4b": ("threshold_results.csv", "threshold"),
    "mark_4c": ("mark_4c_history.csv", "epoch"),
    "mark_4d": ("reconciled_threshold_results.csv", "threshold"),
    "mark_4e": ("fusion_threshold_results.csv", "threshold"),
}


def _metric_color(metric, value, targets):
    if not targets or metric not in targets:
        return "#4C72B0"
    target = targets[metric]
    passed = (value <= target if metric in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else value >= target)
    return "#2E9E5B" if passed else "#C44E52"


def render_summary_dashboard(phase):
    gate = json.loads((OUT[phase] / f"{phase}_gate_result.json").read_text())
    figure, axes = plt.subplots(2, 2, figsize=(15, 9))
    axes[0, 0].axis("off")
    text_lines = [f"phase: {phase}", f"status: {gate.get('status')}",
                  "decision: %s" % (gate.get("decision") or gate.get("next_notebook")
                                    or gate.get("next_step") or "-")]
    for key in ("test_images_accessed", "manifest_sha256", "next_mark", "next_step"):
        if key in gate:
            text_lines.append(f"{key}: {gate[key]}")
    axes[0, 0].text(0.02, 0.99, "\n".join(text_lines), transform=axes[0, 0].transAxes,
                    va="top", ha="left", fontsize=9, family="monospace")
    axes[0, 0].set_title("Gate metadata", fontsize=11, weight="bold")

    selector = GATE_SELECTOR.get(phase)
    selected = gate.get(selector) if selector else None
    targets = TARGETS_SHEET.get(phase)
    row = None
    if isinstance(selected, list):
        chosen = gate.get("selected_arm") or (selected[0].get("arm") if selected else None)
        for arm in selected:
            if arm.get("arm") == chosen:
                row = arm
    else:
        row = selected
    axes[1, 0].set_title("Selected metrics vs targets (green=pass, red=miss)",
                         fontsize=10, weight="bold")
    if row is not None:
        metric_names = [m for m in CORE_METRICS if m in row]
        if metric_names:
            values = [float(row[m]) for m in metric_names]
            axes[1, 0].bar(np.arange(len(metric_names)), values,
                           color=[_metric_color(m, float(row[m]), targets) for m in metric_names])
            axes[1, 0].axhline(0, color="k", lw=0.8)
            for metric in metric_names:
                if targets and metric in targets:
                    axes[1, 0].axhline(targets[metric], color="gray", lw=0.8, ls="--")
            axes[1, 0].set_xticks(np.arange(len(metric_names)))
            axes[1, 0].set_xticklabels(metric_names, rotation=30, ha="right", fontsize=8)
            axes[1, 0].set_ylabel("value")
            if isinstance(selected, list) and row.get("arm"):
                axes[1, 0].set_title(f"Selected arm: {row['arm']} vs targets",
                                     fontsize=10, weight="bold")
        else:
            axes[1, 0].axis("off")
            axes[1, 0].text(0.5, 0.5, "No core-metric table in gate selector",
                            ha="center", va="center")
    else:
        axes[1, 0].axis("off")
        axes[1, 0].text(0.5, 0.5, "No selector in gate JSON", ha="center", va="center")

    csv_name, x_col = TREND_CSV.get(phase, (None, None))
    trend_path = (OUT[phase] / csv_name) if csv_name else None
    if trend_path is not None and trend_path.is_file():
        trend = pd.read_csv(trend_path)
        axes[1, 1].set_title(f"Trend: {csv_name} (x={x_col})", fontsize=10, weight="bold")
        if phase in ("mark_3", "mark_4c"):
            group_col = "configuration" if phase == "mark_3" else "arm"
            y_col = "hard_micro_dice" if phase == "mark_3" else "mean_patient_dice"
            for label, group in trend.groupby(group_col):
                axes[1, 1].plot(group[x_col], group[y_col], marker="o", ms=3, label=str(label))
            axes[1, 1].legend(fontsize=7)
        else:
            y_col = "mean_patient_dice" if "mean_patient_dice" in trend.columns else trend.columns[1]
            axes[1, 1].plot(trend[x_col], trend[y_col], marker="o", ms=3, color="#4C72B0")
        axes[1, 1].set_xlabel(x_col)
        axes[1, 1].set_ylabel("metric")
    else:
        axes[1, 1].axis("off")
        axes[1, 1].text(0.5, 0.5, "Trend CSV not available yet - compute the phase first",
                        ha="center", va="center")

    axes[0, 1].axis("off")
    produced = sorted(p.name for p in OUT[phase].iterdir() if p.is_file())
    inventory = "\n".join(f"- {name}" for name in produced[:20])
    axes[0, 1].text(0.02, 0.99, inventory or "(no data artifacts yet)",
                    transform=axes[0, 1].transAxes, va="top", ha="left", fontsize=8,
                    family="monospace")
    axes[0, 1].set_title(f"Produced artifacts (Evaluation/output/{PHASE_DIR[phase]}/data)",
                         fontsize=10, weight="bold")

    figure.suptitle(f"{phase} - phase summary dashboard", fontsize=15, weight="bold")
    figure.tight_layout(rect=(0, 0, 1, 0.96))
    save_figure(figure, f"{phase}_summary_dashboard.png", phase=phase)
    plt.show()
    print(f"PASS: {phase}_summary_dashboard.png -> {OUT_FIGS[phase]}")

# Provenance + test lock
# ---------------------------------------------------------------------------
from src.framework.data.manifest_dataset import VerifiedManifestDataset

manifest_hash = sha256_file(MANIFEST_PATH)
source_checkpoint_hash = sha256_file(SOURCE_CHECKPOINT)
assert manifest_hash == EXPECTED_MANIFEST_SHA256
assert source_checkpoint_hash == EXPECTED_SOURCE_CHECKPOINT_SHA256

manifest = pd.read_csv(MANIFEST_PATH)
train_manifest = manifest.loc[manifest["split"].eq("train")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
validation_manifest = manifest.loc[manifest["split"].eq("val")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
assert len(manifest) == 58_638
assert len(train_manifest) == 40_667 and train_manifest["volume_id"].nunique() == 104
assert len(validation_manifest) == 10_685 and validation_manifest["volume_id"].nunique() == 13

try:
    VerifiedManifestDataset(MANIFEST_PATH, split="test", root_dir=DATASET_ROOT)
except PermissionError:
    pass
else:
    raise AssertionError("STOP: test split opened without authorization")

print(f"Device: {DEVICE} | REUSE_CACHES={REUSE_CACHES} | REUSE_HISTORY={REUSE_HISTORY}")
print("PASS: provenance, split geometry, and test lock verified.")
print(f"Outputs: {SHARED_OUTPUT_ROOT}")

Device: cuda | REUSE_CACHES=True | REUSE_HISTORY=True
PASS: provenance, split geometry, and test lock verified.
Outputs: D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output


# Part 3 — Mark 3: Two-Stage Multi-Window Training-ROI & Overfit Gate

**Original:** `mark 1/mark_3_two_stage_multiwindow_overfit.ipynb`

## Question

Can the frozen pipeline (prediction-only liver ROI + source-HU windows) overfit a tiny stratified
cohort — proving the ROI geometry and input representation carry the signal — and which channel
configuration is the simplest one that passes?

## Key finding (reproduced)

- Training ROI gate: **PASSED** (100% tumor-pixel and positive-slice containment, 0 empty ROIs).
- Overfit gate: **PASSED** with `broad_1ch` — hard micro-Dice **0.9006** at epoch 17, 0% positive
  predicted-empty. The 2-channel and 3-channel arms were not simpler, so the 1-channel input won.

## Contract

- Training ROI rule: liver threshold 0.50, `largest_3d`, padding 16 (frozen from Mark 2).
- Overfit gate: hard micro-Dice ≥ 0.90, positive predicted-empty = 0%, finite loss/gradients,
  round-trip Dice ≥ 0.98, no test access.
- 16 slices (4 per size quartile) shared by all channel arms.

### 3.1 Verify Mark 2 authorization and load the frozen liver model

In [2]:
mark2_gate = require_upstream_gate("mark_2")
assert mark2_gate["status"] == "mark_2_feasibility_complete"
assert mark2_gate["roi_hard_containment_gate_passed"] is True
assert mark2_gate["roi_efficiency_gate_passed"] is True
assert mark2_gate["test_images_accessed"] is False
assert float(mark2_gate["selected_roi_configuration"]["liver_threshold"]) == 0.50
assert int(mark2_gate["selected_roi_configuration"]["padding"]) == 16
assert mark2_gate["selected_roi_configuration"]["component_mode"] == "largest_3d"

from src.framework.models.mobilenetv2_unet import MobileNetV2UNet

checkpoint = torch.load(SOURCE_CHECKPOINT, map_location="cpu", weights_only=False)
assert int(checkpoint["epoch"]) == 8
liver_model = MobileNetV2UNet(in_channels=1, out_channels=2, pretrained=False)
liver_model.load_state_dict(checkpoint["model_state"], strict=True)
liver_model.to(DEVICE).eval()
with torch.inference_mode():
    probe = liver_model(torch.zeros(1, 1, 256, 256, device=DEVICE))
assert probe.shape == (1, 2, 256, 256)
print("PASS: frozen liver model loaded; Mark 2 authorization confirmed.")

PASS: frozen liver model loaded; Mark 2 authorization confirmed.


### 3.2 Freeze training ROIs (reuse frozen manifest or rebuild)

In [3]:
import shutil
from scipy import ndimage

ORIG_ROI_MANIFEST = MARK1_DIR / "mark_3_outputs" / "training_roi_manifest.csv"
TRAIN_ROI_PATH = OUT["mark_3"] / "training_roi_manifest.csv"


def load_normalized_png(path):
    with Image.open(path) as handle:
        image = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
    return image_robust_normalize(image)


def largest_component(mask):
    labels, count = ndimage.label(mask, structure=np.ones((3, 3, 3), dtype=np.uint8))
    if count == 0:
        return mask
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0
    return labels == sizes.argmax()


def padded_bbox(mask, padding):
    if not mask.any():
        return None
    _, ys, xs = np.where(mask)
    return (max(int(ys.min()) - padding, 0), min(int(ys.max()) + 1 + padding, 256),
            max(int(xs.min()) - padding, 0), min(int(xs.max()) + 1 + padding, 256))


def predict_volume_liver(group):
    probabilities = []
    paths = [DATASET_ROOT / path for path in group["image_path"]]
    with torch.inference_mode():
        for start in range(0, len(paths), 24):
            images = np.stack([load_normalized_png(path) for path in paths[start:start + 24]])
            batch = torch.from_numpy(images[:, None]).float().to(DEVICE)
            probabilities.append(torch.sigmoid(liver_model(batch))[:, 0].cpu().numpy())
    return np.concatenate(probabilities, axis=0)


def score_box(group, box):
    tumor_total = tumor_inside = positive_total = positive_inside = 0
    for row in group.itertuples(index=False):
        with Image.open(DATASET_ROOT / row.tumor_mask_path) as handle:
            tumor = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
        tumor_pixels = int(tumor.sum())
        tumor_total += tumor_pixels
        if tumor_pixels:
            positive_total += 1
        if box is not None:
            y0, y1, x0, x1 = box
            inside = int(tumor[y0:y1, x0:x1].sum())
            tumor_inside += inside
            positive_inside += int(tumor_pixels > 0 and inside > 0)
    return {"tumor_pixel_containment": tumor_inside / tumor_total if tumor_total else np.nan,
            "positive_slice_containment": positive_inside / positive_total if positive_total else np.nan,
            "tumor_pixels": tumor_total, "positive_slices": positive_total}


if REUSE_HISTORY and ORIG_ROI_MANIFEST.is_file():
    shutil.copy2(ORIG_ROI_MANIFEST, TRAIN_ROI_PATH)
    training_rois = pd.read_csv(TRAIN_ROI_PATH)
    print(f"REUSE: training ROI manifest copied ({len(training_rois)} patients).")
else:
    roi_rows = []
    for number, (volume_id, group) in enumerate(
            train_manifest.groupby("volume_id", sort=True), start=1):
        probabilities = predict_volume_liver(group)
        liver_mask = largest_component(probabilities >= 0.50)
        box = padded_bbox(liver_mask, 16)
        scores = score_box(group, box)
        roi_rows.append({"volume_id": int(volume_id), "roi_empty": box is None,
                         "y0": box[0] if box else np.nan, "y1": box[1] if box else np.nan,
                         "x0": box[2] if box else np.nan, "x1": box[3] if box else np.nan,
                         "crop_area_ratio": (((box[1] - box[0]) * (box[3] - box[2])) / (256 * 256)
                                             if box else 0.0), **scores})
        if number % 10 == 0 or number == 104:
            print(f"ROI patients processed: {number}/104")
    training_rois = pd.DataFrame(roi_rows)
    training_rois.to_csv(TRAIN_ROI_PATH, index=False)
    print(f"REBUILD: generated {len(training_rois)} training ROIs.")

display(training_rois.describe(include="all"))

REUSE: training ROI manifest copied (104 patients).


,volume_id,roi_empty,y0,y1,x0,x1,crop_area_ratio,tumor_pixel_containment,positive_slice_containment,tumor_pixels,positive_slices
count,104.000000,104,104.000000,104.000000,104.000000,104.000000,104.000000,96.0,96.0,104.000000,104.000000
unique,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,104,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,51.500000,NaN,32.913462,206.538462,55.432692,211.307692,0.415163,1.0,1.0,24898.682692,47.403846
std,30.166206,NaN,26.313683,30.419374,13.427088,15.593798,0.077756,0.0,0.0,61054.906934,56.451288
min,0.000000,NaN,0.000000,156.000000,15.000000,166.000000,0.205261,1.0,1.0,0.000000,0.000000
25%,25.750000,NaN,11.000000,180.000000,46.000000,201.750000,0.373051,1.0,1.0,645.500000,10.000000
50%,51.500000,NaN,23.500000,198.500000,58.000000,212.000000,0.420425,1.0,1.0,3080.000000,25.000000
75%,77.250000,NaN,58.250000,237.250000,64.000000,222.250000,0.462467,1.0,1.0,13571.750000,63.250000


### 3.3 Training ROI gate + audit

In [4]:
positive_roi_patients = training_rois.loc[training_rois["tumor_pixels"].gt(0)]
training_roi_gate = {
    "minimum_tumor_pixel_containment": float(positive_roi_patients["tumor_pixel_containment"].min()),
    "minimum_positive_slice_containment": float(positive_roi_patients["positive_slice_containment"].min()),
    "empty_training_rois": int(training_rois["roi_empty"].sum()),
    "median_crop_area_ratio": float(training_rois["crop_area_ratio"].median()),
    "maximum_crop_area_ratio": float(training_rois["crop_area_ratio"].max()),
}
training_roi_gate["passed"] = bool(
    training_roi_gate["minimum_tumor_pixel_containment"] >= 0.99
    and training_roi_gate["minimum_positive_slice_containment"] >= 0.99
    and training_roi_gate["empty_training_rois"] == 0)
(OUT["mark_3"] / "training_roi_gate.json").write_text(json.dumps(training_roi_gate, indent=2))

figure, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(training_rois["crop_area_ratio"], bins=20, color="#2878B5")
axes[0].axvline(0.60, linestyle="--", color="#444444")
axes[0].set_title("Training ROI area"); axes[0].set_xlabel("Crop-area ratio")
axes[1].hist(positive_roi_patients["tumor_pixel_containment"],
             bins=np.linspace(0.95, 1.0, 21), color="#4E9F3D")
axes[1].axvline(0.99, linestyle="--", color="#444444")
axes[1].set_title("Tumor-pixel containment")
axes[2].scatter(training_rois["crop_area_ratio"],
                training_rois["tumor_pixel_containment"].fillna(1.0), alpha=0.7)
axes[2].axhline(0.99, linestyle="--", color="#444444")
axes[2].set_title("Containment versus crop burden"); axes[2].set_xlabel("Crop-area ratio")
figure.suptitle("Frozen training ROI audit", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_3"] / "training_roi_audit.png", dpi=170, bbox_inches="tight")
plt.show()
display(pd.DataFrame([training_roi_gate]).T.rename(columns={0: "value"}))
assert training_roi_gate["passed"], "STOP: training ROI gate failed."

C:\Users\alanm\AppData\Local\Temp\ipykernel_3204\3483587942.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,value
minimum_tumor_pixel_containment,1.0
minimum_positive_slice_containment,1.0
empty_training_rois,0
median_crop_area_ratio,0.420425
maximum_crop_area_ratio,0.606201
passed,True


### 3.4 Select the fixed 16-slice stratified cohort

In [5]:
ORIG_SELECTED = MARK1_DIR / "mark_3_outputs" / "overfit_selected_slices.csv"
SELECTED_PATH = OUT["mark_3"] / "overfit_selected_slices.csv"
if REUSE_HISTORY and ORIG_SELECTED.is_file():
    shutil.copy2(ORIG_SELECTED, SELECTED_PATH)
    selected_frame = pd.read_csv(SELECTED_PATH)
    print(f"REUSE: cohort copied ({len(selected_frame)} slices).")
else:
    positive = train_manifest.loc[train_manifest["tumor_pixels"].gt(0)].copy()
    positive["size_quartile"] = pd.qcut(positive["tumor_pixels"], 4,
                                        labels=["Q1", "Q2", "Q3", "Q4"])
    rng = np.random.default_rng(SEED)
    selected_rows, used_patients = [], set()
    for quartile in ["Q1", "Q2", "Q3", "Q4"]:
        candidates = positive.loc[positive["size_quartile"].eq(quartile)].copy()
        candidates = candidates.sample(frac=1.0, random_state=SEED)
        chosen = []
        for row in candidates.itertuples(index=False):
            if row.volume_id not in used_patients or len(chosen) >= 3:
                chosen.append(row)
                used_patients.add(row.volume_id)
            if len(chosen) == 4:
                break
        selected_rows.extend(chosen)
    selected_frame = pd.DataFrame([row._asdict() for row in selected_rows])
    selected_frame.to_csv(SELECTED_PATH, index=False)
    print(f"REBUILD: selected {len(selected_frame)} slices.")

assert len(selected_frame) == 16 and selected_frame["volume_id"].nunique() >= 8
display(selected_frame[["sample_id", "volume_id", "slice_index", "tumor_pixels", "size_quartile"]])

REUSE: cohort copied (16 slices).


,sample_id,volume_id,slice_index,tumor_pixels,size_quartile
0,v084_s0632,84,632,18,Q1
1,v049_s0158,49,158,41,Q1
2,v037_s0092,37,92,42,Q1
3,v008_s0391,8,391,40,Q1
4,v066_s0078,66,78,179,Q2
5,v033_s0049,33,49,189,Q2
6,v029_s0058,29,58,67,Q2
7,v007_s0365,7,365,194,Q2
8,v088_s0432,88,432,699,Q3
9,v056_s0150,56,150,537,Q3


### 3.5 Build source-HU ROI tensors and verify round-trip geometry

In [6]:
import nibabel as nib

roi_index = training_rois.set_index("volume_id")
volume_cache = {}


def source_hu_slice(row):
    volume_id = int(row.volume_id)
    if volume_id not in volume_cache:
        volume_cache[volume_id] = nib.load(str(row.source_volume_path))
    return np.asanyarray(volume_cache[volume_id].dataobj[:, :, int(row.slice_index)]).astype(np.float32)


WINDOWS_3 = {"broad": (-160.0, 240.0), "liver": (-20.0, 140.0), "narrow": (20.0, 120.0)}
CHANNEL_CONFIGS = {"broad_1ch": ["broad"], "broad_liver_2ch": ["broad", "liver"],
                   "broad_liver_narrow_3ch": ["broad", "liver", "narrow"]}

tensors = {name: [] for name in CHANNEL_CONFIGS}
targets = []
roundtrip_rows, preview_rows = [], []
for row in selected_frame.itertuples(index=False):
    box_row = roi_index.loc[int(row.volume_id)]
    y0, y1, x0, x1 = [int(box_row[key]) for key in ("y0", "y1", "x0", "x1")]
    hu_native = source_hu_slice(row)
    channels_256 = {name: resize_float(window_hu(hu_native, (lower, upper)))
                    for name, (lower, upper) in WINDOWS_3.items()}
    with Image.open(DATASET_ROOT / row.tumor_mask_path) as handle:
        tumor_full = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
    target_roi = resize_mask(tumor_full[y0:y1, x0:x1])
    targets.append(target_roi[None].astype(np.float32))
    for config_name, channel_names in CHANNEL_CONFIGS.items():
        crop_channels = [resize_float(channels_256[name][y0:y1, x0:x1])
                         for name in channel_names]
        tensors[config_name].append(np.stack(crop_channels))
    restored_crop = resize_mask(target_roi, size=(x1 - x0, y1 - y0))
    restored = np.zeros((256, 256), dtype=bool)
    restored[y0:y1, x0:x1] = restored_crop
    intersection = int((restored & tumor_full).sum())
    roundtrip_rows.append({"sample_id": row.sample_id, "volume_id": int(row.volume_id),
                           "roundtrip_dice": float((2 * intersection + 1e-6)
                                                   / (restored.sum() + tumor_full.sum() + 1e-6)),
                           "box": [y0, y1, x0, x1]})
    preview_rows.append((row, channels_256, tumor_full, (y0, y1, x0, x1)))

targets = torch.from_numpy(np.stack(targets)).float()
tensors = {name: torch.from_numpy(np.stack(values)).float() for name, values in tensors.items()}
roundtrip_metrics = pd.DataFrame(roundtrip_rows)
roundtrip_metrics.to_csv(OUT["mark_3"] / "roundtrip_geometry_metrics.csv", index=False)
assert roundtrip_metrics["roundtrip_dice"].min() >= 0.98
for name, values in tensors.items():
    assert values.shape[0] == 16 and values.shape[2:] == (256, 256) and torch.isfinite(values).all()
print("PASS: ROI tensors and full-image round-trip geometry verified.")
print(roundtrip_metrics.describe())

figure, axes = plt.subplots(4, 5, figsize=(18, 14))
for row_axes, (row, channels, tumor, box) in zip(axes, preview_rows[:4]):
    y0, y1, x0, x1 = box
    panels = [(channels["broad"], "Broad full"), (channels["liver"], "Liver full"),
              (channels["narrow"], "Narrow full"),
              (resize_float(channels["broad"][y0:y1, x0:x1]), "Broad ROI"),
              (resize_mask(tumor[y0:y1, x0:x1]), "Tumor ROI")]
    for axis, (panel, title) in zip(row_axes, panels):
        axis.imshow(panel, cmap="gray", vmin=0, vmax=1)
        axis.set_title(title); axis.axis("off")
    row_axes[0].set_ylabel(f"{row.sample_id}\n{row.size_quartile}", fontsize=9)
figure.suptitle("Source-HU ROI tensor audit", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_3"] / "roi_tensor_audit.png", dpi=170, bbox_inches="tight")
plt.show()

PASS: ROI tensors and full-image round-trip geometry verified.
       volume_id  roundtrip_dice
count  16.000000            16.0
mean   47.687500             1.0
std    29.279615             0.0
min     4.000000             1.0
25%    28.500000             1.0
50%    44.000000             1.0
75%    71.250000             1.0
max    93.000000             1.0


C:\Users\alanm\AppData\Local\Temp\ipykernel_3204\1177715212.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.6 Warm start, stable loss, and overfit metrics

In [7]:
from src.framework.losses.focal_dice import FocalDiceLoss

source_state = checkpoint["model_state"]
loss_function = FocalDiceLoss(focal_alpha=0.75, focal_gamma=2.0,
                              focal_weight=0.5, dice_weight=0.5)


def build_tumor_model(in_channels):
    model = MobileNetV2UNet(in_channels=in_channels, out_channels=1, pretrained=False)
    target_state = model.state_dict()
    for key, value in source_state.items():
        if key in target_state and target_state[key].shape == value.shape:
            target_state[key] = value.clone()
    source_first = source_state["enc_0.0.weight"]
    target_state["enc_0.0.weight"] = source_first.repeat(1, in_channels, 1, 1) / in_channels
    target_state["final.weight"] = source_state["final.weight"][1:2].clone()
    target_state["final.bias"] = source_state["final.bias"][1:2].clone()
    model.load_state_dict(target_state, strict=True)
    return model.to(DEVICE)


def freeze_batchnorm_running_stats(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()


def hard_metrics(logits, truth):
    prediction = torch.sigmoid(logits) >= 0.50
    target = truth >= 0.5
    intersection = int((prediction & target).sum())
    predicted = int(prediction.sum())
    true = int(target.sum())
    dice = (2 * intersection + 1e-6) / (predicted + true + 1e-6)
    pred_per_slice = prediction.sum(dim=(1, 2, 3))
    true_per_slice = target.sum(dim=(1, 2, 3))
    positive_empty = int(((true_per_slice > 0) & (pred_per_slice == 0)).sum())
    return {"hard_micro_dice": float(dice),
            "positive_predicted_empty_pct": 100 * positive_empty / len(truth)}

### 3.7 Overfit ablation (reuse frozen history or retrain)

In [8]:
ORIG_HIST = MARK1_DIR / "mark_3_outputs" / "overfit_history.csv"
ORIG_COMP = MARK1_DIR / "mark_3_outputs" / "overfit_channel_comparison.csv"
HIST_PATH = OUT["mark_3"] / "overfit_history.csv"
COMP_PATH = OUT["mark_3"] / "overfit_channel_comparison.csv"

if REUSE_HISTORY and ORIG_HIST.is_file():
    shutil.copy2(ORIG_HIST, HIST_PATH)
    shutil.copy2(ORIG_COMP, COMP_PATH)
    for checkpoint_file in (MARK1_DIR / "mark_3_outputs").glob("*_overfit.pth"):
        shutil.copy2(checkpoint_file, OUT["mark_3"] / checkpoint_file.name)
    history_frame = pd.read_csv(HIST_PATH)
    comparison = pd.read_csv(COMP_PATH)
    trained_models = {}
    print(f"REUSE: overfit history + checkpoints copied"
          f" ({len(history_frame)} rows).")
else:
    histories, final_metrics, trained_models = [], [], {}
    for config_name, input_tensor in tensors.items():
        print(f"\n=== {config_name} ===")
        model = build_tumor_model(input_tensor.shape[1])
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
        loader_generator = torch.Generator().manual_seed(SEED)
        loader = DataLoader(TensorDataset(input_tensor, targets), batch_size=4,
                            shuffle=True, generator=loader_generator, num_workers=0)
        best_dice = 0.0
        for epoch in range(1, 161):
            model.train()
            freeze_batchnorm_running_stats(model)
            epoch_loss, gradient_finite = 0.0, True
            for images, truth in loader:
                images, truth = images.to(DEVICE), truth.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                logits = model(images)
                loss = loss_function(logits, truth)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Non-finite loss for {config_name} epoch {epoch}")
                loss.backward()
                gradient_finite &= all(p.grad is None or torch.isfinite(p.grad).all()
                                       for p in model.parameters())
                if not gradient_finite:
                    raise FloatingPointError("Non-finite gradient.")
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
                epoch_loss += float(loss) * len(images)
            model.eval()
            with torch.inference_mode():
                metrics = hard_metrics(model(input_tensor.to(DEVICE)), targets.to(DEVICE))
            histories.append({"configuration": config_name,
                              "channels": int(input_tensor.shape[1]), "epoch": epoch,
                              "loss": epoch_loss / len(input_tensor),
                              "gradient_finite": gradient_finite, **metrics})
            best_dice = max(best_dice, metrics["hard_micro_dice"])
            if epoch == 1 or epoch % 10 == 0:
                print(f"epoch={epoch:03d} loss={histories[-1]['loss']:.4f} "
                      f"dice={metrics['hard_micro_dice']:.4f} "
                      f"empty={metrics['positive_predicted_empty_pct']:.1f}%")
            if metrics["hard_micro_dice"] >= 0.90 and metrics["positive_predicted_empty_pct"] == 0:
                break
        trained_models[config_name] = model.cpu()
        final_metrics.append({"configuration": config_name,
                              "channels": int(input_tensor.shape[1]),
                              "epochs_completed": epoch, "best_hard_micro_dice": best_dice,
                              "final_hard_micro_dice": metrics["hard_micro_dice"],
                              "positive_predicted_empty_pct": metrics["positive_predicted_empty_pct"],
                              "passed": bool(metrics["hard_micro_dice"] >= 0.90
                                             and metrics["positive_predicted_empty_pct"] == 0)})
        torch.save({"model_state": model.state_dict(), "configuration": config_name,
                    "channels": CHANNEL_CONFIGS[config_name],
                    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
                    "source_checkpoint_sha256": EXPECTED_SOURCE_CHECKPOINT_SHA256},
                   OUT["mark_3"] / f"{config_name}_overfit.pth")
    history_frame = pd.DataFrame(histories)
    comparison = pd.DataFrame(final_metrics)
    history_frame.to_csv(HIST_PATH, index=False)
    comparison.to_csv(COMP_PATH, index=False)
    print("REBUILD: overfit ablation completed.")

display(comparison)

REUSE: overfit history + checkpoints copied (79 rows).


,configuration,channels,epochs_completed,best_hard_micro_dice,final_hard_micro_dice,positive_predicted_empty_pct,passed
0,broad_1ch,1,17,0.900557,0.900557,0.0,True
1,broad_liver_2ch,2,20,0.903859,0.903859,0.0,True
2,broad_liver_narrow_3ch,3,42,0.900470,0.900470,0.0,True


### 3.8 Convergence dashboard, selection, and prediction review

In [9]:
passing = comparison.loc[comparison["passed"]].sort_values(
    ["channels", "epochs_completed", "final_hard_micro_dice"],
    ascending=[True, True, False])
selected_configuration = (passing.iloc[0] if not passing.empty else
                          comparison.sort_values("best_hard_micro_dice", ascending=False).iloc[0])

figure, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for name, group in history_frame.groupby("configuration"):
    axes[0].plot(group["epoch"], group["loss"], label=name)
    axes[1].plot(group["epoch"], group["hard_micro_dice"], label=name)
    axes[2].plot(group["epoch"], group["positive_predicted_empty_pct"], label=name)
axes[0].set_title("Overfit loss")
axes[1].axhline(0.90, linestyle="--", color="#444444"); axes[1].set_title("Hard micro-Dice")
axes[2].axhline(0, linestyle="--", color="#444444")
axes[2].set_title("Positive predicted-empty (%)")
for axis in axes:
    axis.set_xlabel("Epoch"); axis.legend(fontsize=8)
figure.suptitle("Controlled channel-ablation overfit", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_3"] / "overfit_convergence_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()
print("Selected:", selected_configuration.to_dict())

# Prediction review (loads saved checkpoint for the selected config)
selected_name = selected_configuration["configuration"]
if selected_name in trained_models:
    selected_model = trained_models[selected_name].to(DEVICE).eval()
else:
    saved = torch.load(OUT["mark_3"] / f"{selected_name}_overfit.pth",
                       map_location="cpu", weights_only=False)
    selected_model = MobileNetV2UNet(in_channels=CHANNEL_CONFIGS[selected_name].__len__(),
                                     out_channels=1, pretrained=False)
    selected_model.load_state_dict(saved["model_state"], strict=True)
    selected_model.to(DEVICE).eval()
selected_inputs = tensors[selected_name].to(DEVICE)
with torch.inference_mode():
    selected_probabilities = torch.sigmoid(selected_model(selected_inputs)).cpu()
predictions = selected_probabilities >= 0.50

figure, axes = plt.subplots(4, 4, figsize=(14, 14))
for row_axes, index in zip(axes, [0, 4, 8, 12]):
    panels = [(selected_inputs[index, 0].cpu(), "Broad ROI"),
              (targets[index, 0], "Expected tumor"),
              (selected_probabilities[index, 0], "Tumor probability"),
              (predictions[index, 0], "Generated tumor")]
    for axis, (panel, title) in zip(row_axes, panels):
        axis.imshow(panel, cmap="magma" if "probability" in title else "gray", vmin=0, vmax=1)
        axis.set_title(title); axis.axis("off")
    row_axes[0].set_ylabel(selected_frame.iloc[index]["sample_id"], fontsize=8)
figure.suptitle(f"Overfit predictions — {selected_name}", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_3"] / "overfit_prediction_review.png", dpi=170, bbox_inches="tight")
plt.show()

C:\Users\alanm\AppData\Local\Temp\ipykernel_3204\2041244626.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Selected: {'configuration': 'broad_1ch', 'channels': 1, 'epochs_completed': 17, 'best_hard_micro_dice': 0.9005570034263234, 'final_hard_micro_dice': 0.9005570034263234, 'positive_predicted_empty_pct': 0.0, 'passed': True}


C:\Users\alanm\AppData\Local\Temp\ipykernel_3204\2041244626.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.9 Write the Mark 3 gate

In [10]:
overfit_passed = bool(selected_configuration["passed"])
geometry_passed = bool(roundtrip_metrics["roundtrip_dice"].min() >= 0.98)
full_gate_passed = bool(training_roi_gate["passed"] and overfit_passed and geometry_passed)

m3_gate = {
    "status": "mark_3_overfit_pass" if full_gate_passed else "mark_3_overfit_fail",
    "training_roi_gate_passed": bool(training_roi_gate["passed"]),
    "overfit_gate_passed": overfit_passed,
    "geometry_gate_passed": geometry_passed,
    "selected_configuration": selected_configuration.to_dict(),
    "selected_channels": CHANNEL_CONFIGS[selected_configuration["configuration"]],
    "minimum_training_tumor_containment": training_roi_gate["minimum_tumor_pixel_containment"],
    "minimum_roundtrip_dice": float(roundtrip_metrics["roundtrip_dice"].min()),
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "source_checkpoint_sha256": EXPECTED_SOURCE_CHECKPOINT_SHA256,
    "test_images_accessed": False,
    "decision": ("PROCEED_TO_3_TO_5_EPOCH_TWO_STAGE_VALIDATION_SMOKE" if full_gate_passed
                 else "STOP_AND_REPAIR_ROI_GEOMETRY_OR_INPUT_REPRESENTATION"),
    "next_notebook": "mark_4_two_stage_validation_smoke" if full_gate_passed else "mark_3_revision",
}
(OUT["mark_3"] / "mark_3_gate_result.json").write_text(json.dumps(m3_gate, indent=2))
display(pd.DataFrame([m3_gate]).T.rename(columns={0: "value"}))
print(m3_gate["decision"])

# ---- Reproduction check against the original gate ----
orig_m3 = json.loads((MARK1_DIR / "mark_3_outputs" / "mark_3_gate_result.json").read_text())
orig_sel = orig_m3["selected_configuration"]
recomputed_sel = m3_gate["selected_configuration"]
diffs = {k: abs(float(recomputed_sel[k]) - float(orig_sel[k])) for k in
         ["best_hard_micro_dice", "final_hard_micro_dice", "positive_predicted_empty_pct"]}
print("Mark 3 reproduction check:", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 3 overfit gate drifted from the original!"
assert m3_gate["status"] == orig_m3["status"]
print("PASS: Mark 3 gate matches the original mark_3_gate_result.json.")

,value
status,mark_3_overfit_pass
training_roi_gate_passed,True
overfit_gate_passed,True
geometry_gate_passed,True
selected_configuration,"{'configuration': 'broad_1ch', 'channels': 1, ..."
selected_channels,[broad]
minimum_training_tumor_containment,1.0
minimum_roundtrip_dice,1.0
manifest_sha256,575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63...
source_checkpoint_sha256,9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c...


PROCEED_TO_3_TO_5_EPOCH_TWO_STAGE_VALIDATION_SMOKE
Mark 3 reproduction check: {'best_hard_micro_dice': 0.0, 'final_hard_micro_dice': 0.0, 'positive_predicted_empty_pct': 0.0}
PASS: Mark 3 gate matches the original mark_3_gate_result.json.


In [11]:

# ---- Standard phase summary dashboard (centralized visualization) ----
render_summary_dashboard("mark_3")


PASS: mark_3_summary_dashboard.png -> D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\03_mark_3\figures


C:\Users\alanm\AppData\Local\Temp\ipykernel_3204\286297166.py:384: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# ---------------------------------------------------------------------------
# Publish key Mark 3 artifacts to the shared/legacy folder other code reads.
#
# External notebooks (e.g. `mark 1 (part 2)/step_00_model_improvement/
# step_1_smoke/step_1_smoke.ipynb`) hardcode artifact paths under
# `mark 1/mark_3_outputs/`. After every Mark 3 run the freshly produced
# artifacts are mirrored there so those code files keep working. The values
# are recomputed from frozen inputs and verified against the original gates
# (reproduction check above), so the mirrored files are equivalent.
# ---------------------------------------------------------------------------
import shutil

PUBLISH_DIR = MARK1_DIR / "mark_3_outputs"
PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

published = []
for pattern in ("*.csv", "*.json", "*.pth"):
    for source in sorted(OUT["mark_3"].glob(pattern)):
        shutil.copy2(source, PUBLISH_DIR / source.name)
        published.append(PUBLISH_DIR / source.name)

for required in ("training_roi_manifest.csv", "mark_3_gate_result.json"):
    assert (PUBLISH_DIR / required).is_file(), f"failed to publish {required}"

print(f"PUBLISHED {len(published)} Mark 3 artifacts to {PUBLISH_DIR}:")
for artifact in sorted(published):
    print("  " + str(artifact))

PUBLISHED 10 Mark 3 artifacts to D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_3_outputs:
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_3_outputs\broad_1ch_overfit.pth
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_3_outputs\broad_liver_2ch_overfit.pth
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_3_outputs\broad_liver_narrow_3ch_overfit.pth
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_3_outputs\mark_3_gate_result.json
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_3_outputs\overfit_channel_comparison.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_3_outputs\overfit_history.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_3_outputs\overfit_selected_slices.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_3_outputs\roundtrip_geometry_metrics.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_3_outputs\training_roi_gate.json
  D:\DATA SCIENCE AND ANALYTICS\PROJ